# M. tuberculosis Variant Calling Pipeline

**Pipeline:** `prefetch`/`fasterq-dump` → `BWA-MEM` → `samtools` → `bcftools` → VCF

**Reference:** H37Rv (NC_000962.3) — standard *M. tuberculosis* reference

**Design notes:**
- Resumable: skips any accession that already has a finished VCF in Drive
- Logs successes/failures separately so you always know exactly what happened to every ID

## Step 1: Mount Drive & set up folder structure

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
BASE_DIR = "/content/drive/MyDrive/tb_pipeline3"
BATCH_DIR = f"{BASE_DIR}/batches"
REF_DIR = f"{BASE_DIR}/reference"
VCF_DIR = f"{BASE_DIR}/vcf_output"
LOG_DIR = f"{BASE_DIR}/logs"

for d in [BATCH_DIR, REF_DIR, VCF_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("drive mounted and folders are ready:")
print(f"  Batches:  {BATCH_DIR}")
print(f"  VCFs out: {VCF_DIR}")
print(f"  Logs:     {LOG_DIR}")

drive mounted and folders are ready:
  Batches:  /content/drive/MyDrive/tb_pipeline3/batches
  VCFs out: /content/drive/MyDrive/tb_pipeline3/vcf_output
  Logs:     /content/drive/MyDrive/tb_pipeline3/logs


## Step 2: Install tools

In [3]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
!bash /tmp/miniforge.sh -b -p /usr/local/miniforge

import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!mamba install -y -c bioconda -c conda-forge sra-tools=3.1.1 bwa samtools bcftools parallel >/dev/null
!which prefetch fasterq-dump bwa samtools bcftools parallel

PREFIX=/usr/local/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.5.20-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_19.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_102.conda
Extracting libgcc-15.2.0-he0feb66_19.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.8.1-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_19.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.3-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libsqlite-3.53.1-h0c1763c_0

In [4]:
!fasterq-dump --version
!bwa 2>&1 | head -3
!samtools --version | head -1
!bcftools --version | head -1


fasterq-dump : 3.1.1


Program: bwa (alignment via Burrows-Wheeler transformation)
Version: 0.7.19-r1273
samtools 1.23.1
bcftools 1.23.1


## Step 3: Download & index the H37Rv reference genome

In [5]:
ref_fasta = f"{REF_DIR}/H37Rv.fasta"

if not os.path.exists(ref_fasta):
    print("downloading H37Rv reference (NC_000962.3) from NCBI...")
    !curl -s "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000962.3&rettype=fasta&retmode=text" -o "{ref_fasta}"
    print("done!!!")
else:
    print("reference is already present in the drive, skipping download.")

!head -1 "{ref_fasta}"
!grep -c ">" "{ref_fasta}"
!wc -l "{ref_fasta}"

reference is already present in the drive, skipping download.
>NC_000962.3 Mycobacterium tuberculosis H37Rv, complete genome
1
63024 /content/drive/MyDrive/tb_pipeline3/reference/H37Rv.fasta


In [6]:
#indexing the reference (BWA index + samtools faidx)
if not os.path.exists(f"{ref_fasta}.bwt"):
    print("building BWA index...")
    !bwa index "{ref_fasta}"
else:
    print("BWA index already exists, skipping")

if not os.path.exists(f"{ref_fasta}.fai"):
    !samtools faidx "{ref_fasta}"
else:
    print("samtools faidx index already exists, skipping")

print("Reference ready__")

BWA index already exists, skipping
samtools faidx index already exists, skipping
Reference ready__


## Step 4: worker script


In [7]:
worker_script = f'''#!/bin/bash
# - BWA piped directly into samtools sort (no SAM file written to disk)
# - SRA cache deleted immediately after fasterq-dump
# - FASTQs deleted immediately after alignment
# - BAM deleted after variant calling
# - Atomic write to Drive via .tm
set -uo pipefail

ID="$1"
REF="{ref_fasta}"
VCF_OUT_DIR="{VCF_DIR}"
LOG_DIR="{LOG_DIR}"
SCRATCH="/content/scratch/$ID"
SUCCESS_LOG="$LOG_DIR/success.log"
FAIL_LOG="$LOG_DIR/failed.log"
FINAL_VCF="$VCF_OUT_DIR/${{ID}}.vcf.gz"
TMP_VCF="$VCF_OUT_DIR/.${{ID}}.vcf.gz.tmp"


#this here checks the already completed files
if [ -f "$FINAL_VCF" ]; then
    echo "[$ID] SKIP - already completed"
    exit 0
fi

#Cleaning up stale files from any previous failed attempt on this ID
rm -rf "$SCRATCH"
rm -f "$TMP_VCF" "${{TMP_VCF}}.csi"
mkdir -p "$SCRATCH/tmp" "$VCF_OUT_DIR" "$LOG_DIR"

fail() {{
    echo -e "$ID\\t$1" >> "$FAIL_LOG"
    echo "[$ID] FAILED - $1"
    rm -rf "$SCRATCH"
    rm -f "$TMP_VCF" "${{TMP_VCF}}.csi"
    exit 1
}}

#downloading the data
echo "[$ID] PREFETCH"
prefetch "$ID" -O "$SCRATCH" >/dev/null 2>"$SCRATCH/prefetch.err"
[ $? -ne 0 ] && fail "prefetch failed"

echo "[$ID] FASTERQ-DUMP"
fasterq-dump "$SCRATCH/$ID" \
    -O "$SCRATCH" \
    -t "$SCRATCH/tmp" \
    --split-3 \
    -e 4 \
    >/dev/null 2>"$SCRATCH/fasterq.err"
[ $? -ne 0 ] && fail "fasterq-dump failed"

#deleting the SRA after conversion
rm -rf "$SCRATCH/$ID"
rm -rf "$HOME/ncbi/public/sra/${{ID}}.sra"* 2>/dev/null || true

#detecting the sequence layout
R1="$SCRATCH/${{ID}}_1.fastq"
R2="$SCRATCH/${{ID}}_2.fastq"
SE="$SCRATCH/${{ID}}.fastq"
BAM="$SCRATCH/${{ID}}.sorted.bam"
RAW_VCF="$SCRATCH/${{ID}}.raw.vcf.gz"

if [ -s "$R1" ] && [ -s "$R2" ]; then
    MODE="paired"
elif [ -s "$SE" ]; then
    MODE="single"
else
    fail "no usable FASTQ files found"
fi

echo "[$ID] ALIGN ($MODE)"


#Alignment: BWA piped into samtools sort
RG="@RG\\\\tID:${{ID}}\\\\tSM:${{ID}}"

if [ "$MODE" = "paired" ]; then
    bwa mem -t 2 -R "$RG" "$REF" "$R1" "$R2" 2>"$SCRATCH/bwa.err" \
        | samtools sort -@ 2 -m 1G -o "$BAM" - 2>"$SCRATCH/sort.err"
else
    bwa mem -t 2 -R "$RG" "$REF" "$SE" 2>"$SCRATCH/bwa.err" \
        | samtools sort -@ 2 -m 1G -o "$BAM" - 2>"$SCRATCH/sort.err"
fi
[ $? -ne 0 ] || [ ! -s "$BAM" ] && fail "bwa/samtools sort failed"

#deleting fastq files after alignment
rm -f "$R1" "$R2" "$SE" 2>/dev/null || true

#indexing the BAM file
echo "[$ID] INDEX"
samtools index "$BAM"
[ $? -ne 0 ] && fail "samtools index failed"

#variant calling
echo "[$ID] VARIANT CALLING"
bcftools mpileup -f "$REF" "$BAM" 2>"$SCRATCH/mpileup.err" \
    | bcftools call -mv -Oz -o "$RAW_VCF" 2>"$SCRATCH/call.err"
[ $? -ne 0 ] || [ ! -s "$RAW_VCF" ] && fail "bcftools failed"

bcftools index "$RAW_VCF"
[ $? -ne 0 ] && fail "bcftools index failed"


#deleting the BAM file after variant calling
rm -f "$BAM" "$BAM.bai" 2>/dev/null || true

# atomic copy to drive
# mv across filesystem boundary (local -> Drive FUSE) is NOT atomic.
# Pattern: copy to .tmp first, then mv within Drive (same filesystem = truly atomic).
echo "[$ID] COPY TO DRIVE"
cp "$RAW_VCF" "$TMP_VCF"
cp "${{RAW_VCF}}.csi" "${{TMP_VCF}}.csi"
mv "$TMP_VCF" "$FINAL_VCF"
mv "${{TMP_VCF}}.csi" "${{FINAL_VCF}}.csi"

[ -f "$FINAL_VCF" ] || fail "copy to Drive failed"

echo -e "$ID\\tOK" >> "$SUCCESS_LOG"
echo "[$ID] SUCCESS"

rm -rf "$SCRATCH"
exit 0
'''

with open("/content/worker.sh", "w") as f:
    f.write(worker_script)

import subprocess
subprocess.run(["chmod", "+x", "/content/worker.sh"])

# Verifying syntax
result = subprocess.run(["bash", "-n", "/content/worker.sh"], capture_output=True, text=True)
if result.returncode == 0:
    print("worker.sh written correctly")
else:
    print("SYNTAX ERROR:", result.stderr)

checks = {
    "correct scratch path": 'SCRATCH="/content/scratch/$ID"',
    "set -e removed": "set -uo pipefail",
    "BWA pipe": "samtools sort",
    "Atomic tmp write": ".vcf.gz.tmp",
    "fasterq threads 4": "-e 4",
    "samtools memory": "-m 1G",
}
print("\nVerification:")
with open("/content/worker.sh") as f:
    content = f.read()
for label, pattern in checks.items():
    found = "OK" if pattern in content else "MISSING"
    print(f"  [{found}] {label}")

worker.sh written correctly

Verification:
  [OK] correct scratch path
  [OK] set -e removed
  [OK] BWA pipe
  [OK] Atomic tmp write
  [OK] fasterq threads 4
  [OK] samtools memory


## Step 5: Select Batch

In [8]:
BATCH_NAME = "batch_028"
batch_path = f"{BATCH_DIR}/{BATCH_NAME}"

if not os.path.exists(batch_path):
    raise FileNotFoundError(
        f"could not find {batch_path}\n"
        f"check whether {BATCH_NAME} is uploaded into {BATCH_DIR} in the drive."
    )

with open(batch_path) as f:
    ids = [line.strip() for line in f if line.strip()]

print(f"Batch: {BATCH_NAME}")
print(f"Total IDs in this batch: {len(ids)}")
print(f"First 5: {ids[:5]}")

already_done = [i for i in ids if os.path.exists(f"{VCF_DIR}/{i}.vcf.gz")]
print(f"Already completed (will be skipped): {len(already_done)}")
print(f"Remaining to process: {len(ids) - len(already_done)}")

Batch: batch_028
Total IDs in this batch: 100
First 5: ['ERR8025607', 'ERR8025608', 'ERR8025609', 'ERR8025610', 'ERR8025611']
Already completed (will be skipped): 73
Remaining to process: 27


## Step 6: Run the pipeline

In [9]:
import subprocess
import time

PARALLEL_JOBS = 4
queue_file = "/content/current_queue.txt"
with open(queue_file, "w") as f:
    f.write("\n".join(ids) + "\n")

print(f"starting pipeline for {BATCH_NAME} ({len(ids)} IDs, {PARALLEL_JOBS} at a time)...")
print("=" * 80)

start_time = time.time()

cmd = f"cat {queue_file} | parallel -j {PARALLEL_JOBS} --joblog {LOG_DIR}/{BATCH_NAME}_joblog.tsv /content/worker.sh"

process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line.rstrip())
process.wait()

elapsed = time.time() - start_time
print("=" * 80)
print(f"Batch {BATCH_NAME} run finished in {elapsed/60:.1f} minutes")

starting pipeline for batch_028 (100 IDs, 4 at a time)...
[ERR8025607] SKIP - already completed
[ERR8025608] SKIP - already completed
[ERR8025609] SKIP - already completed
[ERR8025610] SKIP - already completed
[ERR8025611] SKIP - already completed
[ERR8025612] SKIP - already completed
[ERR8025613] SKIP - already completed
[ERR8025614] SKIP - already completed
[ERR8025615] SKIP - already completed
[ERR8025616] SKIP - already completed
[ERR8025617] SKIP - already completed
[ERR8025618] SKIP - already completed
[ERR8025619] SKIP - already completed
[ERR8025620] SKIP - already completed
[ERR8025621] SKIP - already completed
[ERR8025622] SKIP - already completed
[ERR8025623] SKIP - already completed
[ERR8025624] SKIP - already completed
[ERR8025625] SKIP - already completed
[ERR8025626] SKIP - already completed
[ERR8025627] SKIP - already completed
[ERR8025628] SKIP - already completed
[ERR8025629] SKIP - already completed
[ERR8025630] SKIP - already completed
[ERR8025631] SKIP - already co

## Step 7: Check results


In [10]:
success_log = f"{LOG_DIR}/success.log"
fail_log = f"{LOG_DIR}/failed.log"

n_success = 0
n_fail = 0

if os.path.exists(success_log):
    with open(success_log) as f:
        succeeded_ids = set(line.split("\t")[0] for line in f if line.strip())
    n_success = len(succeeded_ids & set(ids))

if os.path.exists(fail_log):
    with open(fail_log) as f:
        failed_lines = [line.strip().split("\t") for line in f if line.strip()]
    failed_this_batch = [(i, reason) for i, reason in failed_lines if i in ids]
    n_fail = len(failed_this_batch)
else:
    failed_this_batch = []

print(f"Batch: {BATCH_NAME}")
print(f"Succeeded: {n_success} / {len(ids)}")
print(f"Failed: {n_fail} / {len(ids)}")

if failed_this_batch:
    print("\nFailed IDs and reasons:")
    for i, reason in failed_this_batch:
        print(f"  {i}: {reason}")

vcf_count = len([f for f in os.listdir(VCF_DIR) if f.endswith('.vcf.gz')])
print(f"\nTotal VCF files in {VCF_DIR}: {vcf_count}")

Batch: batch_028
Succeeded: 100 / 100
Failed: 0 / 100

Total VCF files in /content/drive/MyDrive/tb_pipeline3/vcf_output: 100


Once you're ready to convert all your VCFs into a single CSV for the ML step, that's a separate, fairly quick step (e.g. `bcftools merge` + extracting variant matrix with `scikit-allel` or `cyvcf2`) — happy to help with that once VCF generation is underway.

In [11]:
import shutil

shutil.make_archive('/content/vcf_output_batch028', 'zip', '/content/drive/MyDrive/tb_pipeline3/vcf_output')
print("Zip created at /content/vcf_output_batch028.zip")

Zip created at /content/vcf_output_batch028.zip


In [12]:
from google.colab import files
files.download('/content/vcf_output_batch028.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>